# Task 1: rating prediction (MSE ↓, 0-ratings skipped)

**Shipped:** bias model α + β_u + β_i (ALS), `TASK1_BIAS_PARAMS` = reg_u 1, reg_i 5, 25 epochs,
trained on nonzero rows. **Val 0.5941 | public 0.5917 | tier 2.5** (3.0 ≤ 0.58, 3.5 ≤ 0.56).

**Data facts** (checks below): no test timestamps; 0% of test pairs in train; 1.48% zero rows
(23.7% of users have one); 1.3% of test rows from single-rating users.

**Open question:** leaderboard cluster at ~0.52 (tier 4.0); cause unknown. Ask a TA.

In [ ]:
%load_ext autoreload
%autoreload 2
 
import itertools
import numpy as np
import pandas as pd
from surprise import SVD, Dataset, Reader
from tqdm.auto import tqdm
from scipy.sparse import csr_matrix
from assignment_1.modules.models import fit_bias, predict_bias,fit_item_cosine
from assignment_1.modules.util import (
    calc_mse,
    drop_zero_ratings,
    load_csv,
    random_split,
)

DATA_DIR = "../data"  # notebook runs from notebooks/

In [ ]:
TASK_TRAIN_FILE = "tasks_1_2_train.csv"
TASK_TEST_FILE = "task_1_test.csv"
train = load_csv(TASK_TRAIN_FILE, DATA_DIR)
test = load_csv(TASK_TEST_FILE, DATA_DIR)

SCALE = (1, 5)

## Data checks

In [ ]:
print(f"{len(train):,} rows | {train.user.nunique():,} users | {train.item.nunique():,} items")
print(
    f"test unseen: users {(~test.user.isin(train.user)).mean():.1%}, items {(~test.item.isin(train.item)).mean():.1%}")
print(
    f"test columns: {test.columns.tolist()}  (rating column is empty: the prediction slot)")


def key(d): return d["user"] + "|" + d["item"]


z = train["rating"] == 0
g = train[~z].groupby("user")["rating"].nunique()
print(f"test pairs in train: {key(test).isin(set(key(train))).mean():.2%}")
print(
    f"zero rows: {z.mean():.2%} | users with any zero: {train.loc[z, 'user'].nunique() / train.user.nunique():.1%}")
print(
    f"test rows from single-rating users: {(test.user.map(g) == 1).mean():.1%}")
train.rating.value_counts().sort_index()

1,861,643 rows | 30,909 users | 51,007 items
test unseen: users 0.0%, items 0.0%
test columns: ['user', 'item', 'rating']  (rating column is empty: the prediction slot)
test pairs in train: 0.00%
zero rows: 1.48% | users with any zero: 23.7%
test rows from single-rating users: 1.3%


rating
0     27626
1     36443
2     97748
3    314624
4    641182
5    744020
Name: count, dtype: int64

## Validation split + yardstick
One split for every experiment. It must reproduce the shipped number (0.5941) or the
comparisons below are meaningless.

In [ ]:
# the split that reproduces 0.5941
tr, val = random_split(train, val_frac=0.25, seed=0)
tr, val = drop_zero_ratings(tr), drop_zero_ratings(val)
y_val = val["rating"].to_numpy()

bias_fit = fit_bias(tr, reg_u=1.0, reg_i=5.0, n_epochs=25, rating_scale=SCALE)
base_pred = predict_bias(bias_fit, val.user, val.item)
BASE = calc_mse(y_val, base_pred)
print(f"shipped bias model: {BASE:.4f}  (expect 0.5941)")

shipped bias model: 0.5941  (expect 0.5941)


## Helpers

In [ ]:
ts = Dataset.load_from_df(tr[["user", "item", "rating"]], Reader(
    rating_scale=SCALE)).build_full_trainset()
val_tuples = list(val[["user", "item", "rating"]
                      ].itertuples(index=False, name=None))


def svd_mse(**params) -> float:
    m = SVD(biased=True, random_state=0, **params).fit(ts)
    return calc_mse(y_val, [p.est for p in m.test(val_tuples)])


def run_grid(score, grid: dict) -> pd.DataFrame:
    """score(**params) -> MSE for every combination in grid; sorted, with % vs the shipped model."""
    rows = []
    for values in tqdm(list(itertools.product(*grid.values()))):
        params = dict(zip(grid, values))
        mse = score(**params)
        rows.append({**params, "val_mse": mse,
                    "vs_base_%": (1 - mse / BASE) * 100})
    return pd.DataFrame(rows).sort_values("val_mse").reset_index(drop=True)


def svd_sep(k, reg_b, reg_f, lr=0.005, epochs=20):
    """SVD with separate bias / factor regularization (SGD scale)."""
    return svd_mse(n_factors=k, reg_bu=reg_b, reg_bi=reg_b, reg_pu=reg_f, reg_qi=reg_f,
                   lr_all=lr, n_epochs=epochs)


def bias_mse(reg_u, reg_i, n_epochs=25):
    fit = fit_bias(tr, reg_u=reg_u, reg_i=reg_i,
                   n_epochs=n_epochs, rating_scale=SCALE)
    return calc_mse(y_val, predict_bias(fit, val.user, val.item))

## E1. Bias model λ (shipped)
**Result:** best reg_u 1, reg_i 5, 25 epochs → 0.5941. More epochs: no change.

In [6]:
# run_grid(
#     bias_mse,
#     {
#         "reg_u": [0.3, 1, 3],
#         "reg_i": [2, 5, 10],
#         "n_epochs": [25]
#     }
# )

## E2. SVD with one λ (`reg_all`)
**Result:** best k = 1, reg_all 0.05, 50 epochs → 0.5989; blending with bias did not beat bias alone.

In [7]:
# run_grid(
#     svd_mse,
#     {
#         "n_factors": [1, 10, 50],
#         "reg_all": [0.02, 0.05, 0.1, 0.3],
#         "n_epochs": [20, 50]
#     }
# )

## E3. SVD with separate bias / factor λ
**Result:** best 0.5980 (k 5, reg_b 0.005, reg_f 0.3, lr 0.003, 60 epochs). Still worse than
bias (0.5941). reg_f barely matters → factors contribute ~nothing. Best rows at grid edges
(lowest lr, most epochs, lowest reg_b): extend before concluding.

In [8]:
# run_grid(
#     svd_sep,
#     {
#         "k": [5],
#         "reg_b": [0.005, 0.02],
#         "reg_f": [0.05, 0.1, 0.3],
#         "lr": [0.003, 0.01],
#         "epochs": [20, 60]
#     }
# )

## E4. Are the factors doing anything?
Same SGD settings with factors frozen (reg_f = 100) vs free. Equal → factors add nothing.

In [9]:
pd.Series({
    "shipped bias (ALS)": BASE,
    "SVD factors frozen": svd_sep(5, 0.005, 100, lr=0.003, epochs=60),
    "SVD best (E3)": svd_sep(5, 0.005, 0.3, lr=0.003, epochs=60),
}).round(5)

shipped bias (ALS)    0.59410
SVD factors frozen    0.59802
SVD best (E3)         0.59802
dtype: float64

In [ ]:
err = (base_pred - y_val) ** 2
n_u = val["user"].map(tr["user"].value_counts()).fillna(0).to_numpy()
n_i = val["item"].map(tr["item"].value_counts()).fillna(0).to_numpy()
u_std = val["user"].map(tr.groupby(
    "user")["rating"].std()).fillna(0).to_numpy()


def err_by(x, bins, name):
    g = (pd.DataFrame({"b": pd.cut(x, bins, include_lowest=True), "e": err})
           .groupby("b", observed=True)["e"].agg(["mean", "size", "sum"]))
    g["share_of_error"] = g["sum"] / err.sum()
    print(f"\n{name}\n{g[['mean', 'size', 'share_of_error']].round(3)}")


err_by(n_u, [0, 1, 3, 10, 30, 100, 1e6], "user history size")
err_by(n_i, [0, 1, 3, 10, 30, 100, 1e6], "item popularity")
err_by(u_std, [0, 0.01, 0.5, 0.8, 1.0, 1.3, 5], "user rating std")
err_by(base_pred, [1, 3, 3.5, 4, 4.5, 5], "predicted value")
err_by(y_val, [0.5, 1.5, 2.5, 3.5, 4.5, 5.5], "true rating")


user history size
                     mean    size  share_of_error
b                                                
(-0.001, 1.0]       0.634      16           0.000
(1.0, 3.0]          0.617     176           0.000
(3.0, 10.0]         0.779   26201           0.075
(10.0, 30.0]        0.708   74471           0.194
(30.0, 100.0]       0.629  122315           0.282
(100.0, 1000000.0]  0.519  235290           0.449

item popularity
                     mean    size  share_of_error
b                                                
(1.0, 3.0]          0.617     302           0.001
(3.0, 10.0]         0.581   57782           0.123
(10.0, 30.0]        0.560  131511           0.270
(30.0, 100.0]       0.580  125965           0.268
(100.0, 1000000.0]  0.643  142909           0.337

user rating std
                 mean    size  share_of_error
b                                            
(-0.001, 0.01]  0.135    7324           0.004
(0.01, 0.5]     0.204   56784           0.043
(0.5, 0.8]   

In [ ]:
# run_grid(svd_sep, {"k": [5], "reg_b": [0.005], "reg_f": [0.3], "lr": [0.002, 0.003], "epochs": [60, 120, 200]})

  0%|          | 0/6 [00:00<?, ?it/s]

,k,reg_b,reg_f,lr,epochs,val_mse,vs_base_%
0,5,0.005,0.3,0.002,120,0.597200,-0.522381
1,5,0.005,0.3,0.002,200,0.597498,-0.572540
2,5,0.005,0.3,0.003,120,0.597552,-0.581594
3,5,0.005,0.3,0.003,60,0.598020,-0.660492
4,5,0.005,0.3,0.003,200,0.598050,-0.665457
5,5,0.005,0.3,0.002,60,0.600616,-1.097399


In [ ]:
cos_fit = fit_item_cosine(tr)
res = tr["rating"].to_numpy() - predict_bias(bias_fit,
                                             tr.user, tr.item)    # r_uj − b̂_uj
Rres = csr_matrix((res, (cos_fit.users.get_indexer(tr.user), cos_fit.items.get_indexer(tr.item))),
                  shape=cos_fit.R.shape)


def knn_parts(users, items, chunk=50_000):
    u, i = cos_fit.users.get_indexer(users), cos_fit.items.get_indexer(items)
    num, den = np.zeros(len(u)), np.zeros(len(u))
    for s in range(0, len(u), chunk):
        uu, ii = u[s:s + chunk], i[s:s + chunk]
        ok = (uu >= 0) & (ii >= 0)
        Si = cos_fit.S[ii[ok]]
        num[s:s +
            chunk][ok] = np.asarray(Rres[uu[ok]].multiply(Si).sum(axis=1)).ravel()
        den[s:s + chunk][ok] = np.asarray(cos_fit.R[uu[ok]
                                                    ].multiply(Si).sum(axis=1)).ravel()
    return num, den


num, den = knn_parts(val.user, val.item)
for lam in [0.5, 1, 2, 5, 10]:
    pred = np.clip(base_pred + num / (den + lam), 1, 5)
    print(f"λ={lam:<4} MSE {calc_mse(y_val, pred):.4f}  (bias {BASE:.4f})")

## Next
- E5: extend E3 past its edges (reg_b 0–0.005, lr 0.001–0.003, 60–120 epochs) only if E4 shows factors help.
- E6: implicit feedback (zeros as interactions): residual features first, SVD++ if they help.
- Ask a TA what reaches ~0.52 with course tools.